# Word Embeddings and Linguistic

## Properties of Word Embeddings

### Word Embeddings v.s. Document Vector
- Word vector and document vector:
    - Previosuly we focus on vector representation of documents to predict an outcome variable not deriavble from the text
    - Now we introduce word embeddings, which is a vector representation that predicts the meaning of a word by its neighboring words (local contexts)
    - That is, rather than predicting some metadata, word vectors predict the co-occurence of neighboring words
- One major type of word embedding models is GloVe, denoting Global Vectors for Word Representation
- The arrangment of word vectors as a matrix is called word context matrix
- Word context matrix in comparision with document term matrix:
    - Document term matrix is usually high-dimensional:
        - The number of unique words (column numbers) is huge in a corpus (e.g. > 50000)
        - Besides, most elements in the matrix is 0 (despite common words, most words appear rarely in a document), so the matrix is sparse
    - Word context matrix is low dimensional:
        - The number of elements in the word vectors (columns of word contex matrix) usually = 300
        - The dimension of word vector has no meaning, setting it as 300 is simply becuase it works well
        - Besides, word context matrix is dense, elements usually do not equals 0

### Word Context Matrix
- Matrix definition:
    - Each row $w$ represents a word of the vocabulary
    - Each column $c$ represents a linguistic context in which words can occur
    - A matrix entry $M[w,c]$ quantifies the strength of association between a word and a context in the corpus
- Context window:
    - Popular embeddings (such as GloVe) generally use 5-word or 10-word context window (the context is measured as 5/10 words neighbouring)
- Word vector:
    - Each word vector (row) $M[w,:]$ is a distribution over contexts
    - Different definitions of contexts and different measures of association lead to different types of word vectors
    - These vectors have a spatial interpretation, such that geometric distances between word vectors reflect semantic distances between words

### GloVe Embedding
- A popular way to embed word into dense vectors with semantic meaning is GloVe (Global Vectors for Word Representation) by Pennington et al (2014)
- Input:
    - $C_{ij}$ = local co-occurrence counts between words $i,j ∈ {1,...,n_w}$ within context (co-occurrence) window (usually 5 to 10)
- Output:
    - Word context matrix (word vectors) $w = (w1,...,w_i,...,w_{n_w})^\top$, where ${w_i ∈ (−1,1)}^{n_E}$
    - Where $n_w$ denotes number of words in the corpus
    - And $n_E$ denotes the setted dimension of word vectors (usually 300)
- Algorithm:
    - Learn word vectors $w$ to solve: 
    $$
    \min _{w} \sum_{i,j}f(C_{ij})(w_i^\top w_j - log(C_{ij}))^2
    $$
    - Where $f (·)$ is weighting function to down-weight frequent words:
        - This is because for frequent words ("the", "this", etc.), their cooccurence with other words only provide limited informativeness on semnatic relationship (however, **not none**, so do not remove stop words)
        - For example, ["the", "apple"] and ["the", "rocket"] both cooccurs frequently does not mean that "apple" is similar with "rocket"
        - Also "apple" is not similar with "the"
- Intuition:
    - The model finds word vectors that minimizes suqared difference between dot product of word vectors and log co-occurence
    - Dot product of word vectors: measures correlation between two word vectors
    - Log co-occurence: using log is just becuase it works well 
        - Arora et al (2016) put the PMI here instead of log co-occurence
        - PMI (pointwise mutual information) is simply p(i and j cooccur in the context window)/[p(i occur)*p(j occur)]
    - Therefore, words that co-occur or have a similar set of words that frequently co-occur should have higher similarity in word vector

### Algebra on Word Vectors 
- Once words are represented as vectors, we can use linear algebra to understand the relationships between words
- Similarity:
    - Words that are geometrically close to each other are similar, the standard metric for vector similarity is cosine similarity
    - Cosine similarity: $$cos \theta = \frac{v_1 \cdot v_2}{\|v_1\| \cdot \|v_2\|}$$
- Produce Document Vector:
    - The “continuous bag of words” representation for document $D$ is the weighted average of the vectors $w_i$ for each word $i$ in the document: 
        $$
        D = \sum_{w_i \in D}a_{w_i}w_i
        $$
        - Word vectors $w_i$ constructed using GloVe
        - Document could be sentence, paragraph, section, etc.
    - Arora, Liang, and Ma (2017) provide a “baseline” weight as: $$ a_{w_i} = \frac{\alpha}{\alpha + p_{i}}$$
        - Where $p_i$ is the probability (frequency) of the word and $\alpha = .001$ is a smoothing parameter
        - This smoothed inverse frequency weighted average of the vectors achieve outstanding performance in average
- Analogies:
    - Algebra on word vectors produced by GloVe can also depict conceptual, analogical relationships between words
    - For example, the direction from the vector representing "man" to "woman" is very similar to the direction from "king" to "queen"
    - That is: vec(king) - vec(man) + vec(woman) = vec(queen) 
    - More generally, we can find the analogy of king : man in the woman case as $$ \argmax_{b_2 \in V} cos(b_2, a_2 - a_1 + b_1)$$
        - Where $a_2$ denotes king, $a_1$ denotes male, $b_1$ denotes female
        - And $V$ excludes $(a_1,b_1,a_2)$
    - This method of finding analogies through vector addition / subtraction works better with normalized vectors
    - So that long one vector doesn’t wash out the direction of others

### What do Word Embedding Similarity Capture
- Semantic similarity: words sharing similar attributes, including below relationships
    - Synonymy (car / automobile)
    - Hypernymy (car / vehicle)
    - Co-hyponymy (car / van / truck)
- Semantic relatdness: words semantically associated, including below relationships
    - Function (car / drive)
    - Meronymy (car / tire)
    - Location (car / road)
    - Attribute (car / fast)
- Word embeddings can recover one or both of these relations, depending on how contexts and vectors are constructed:
    - Suppose we want to find similar words to dog
    - In a 2-word window, we will find: cat, horse, fox etc.
    - In a 30-word window, we will find: puppy, pet etc.
    - Hence, small windows pick up substitutable words (e.g. co-hyponymy), while large windows pick up topics (e.g. hypernymy)

### Limitations and Ways to Improve
- In the default model multiple senses of a word are merged and sharing one vector representation:
    - Example: “I like a bird” (verb) and “I am like a bird” (preposition)
    - Can improve embeddings in these cases by attaching the Part-Of-Speech tagging to the word (e.g. “like:verb”, “like:prep”) before training
- The default model only works by word, but “new york" not equal to ”new” + “york”
    - Can tokenize phrases together before training (e.g. using n-grams)

Non-solvable Limitations:
- Black Sheep Problem:
    - The trivial or most obvious features of a word are not mentioned in standard corpora
    - For example, although most sheep are white, we rarely see the phrase “white sheep”
    - As a result, GloVe can tell us sim(black,sheep) > sim(white,sheep), as the likelihood that people mention black sheep > white sheep
    - This is a important limitation when we use embeddings to anayze beliefs/attitudes
    - As we may falsely identify that people tend to believe sheep is black rather than white in common
    - It is not true: it is just because wheet sheep is so common, that people do not mention it **obviously**
- Rare words
    - A word that shows up just once or twice won’t be well-defined (in term of word vector)
- Polysemy: 
    - We get one vector for multiple senses of a word even after attaching POS tag, when different sense of the word have same POS tagging
    - For example, “glass of water” vs “window glass”, both are nouns, but different meaning

## Word Embeddings in Practice

### Tokenizing for Word Embeddings
- Drop capitalization
- Punctuation is optional
- Add special tokens for start of sentence and end of sentence
    - Otherwise cannot tell two words are co-occuring because of semantic context or just because sentences were concatenated
- Don't drop stopwords/function-words
    - Stop words still provide semantic information (though limited)
    - Dropping them increase noise in word embedding
    - Instead of dropping, we allow the model to downweight them
- For out-of-vocab words, substitute a special token or replace with part-of-speech tag

### Pre-trained Word Embedding Model
- In many settings (e.g. a small corpus), it is better to use pre-trained embeddings
- For example, spaCy’s GloVe embeddings:
    - One million vocabulary entries
    - 300-dimensional vectors
    - Trained on the Common Crawl corpus
- We can initialize models with pre-trained embeddings and fine-tune them as needed

### Embedding with Subword Information
- Bojanowski et al (2017) represented each word as a bag of character n-grams
- For example, spicy = (spi, pic, icy)
- They then train the model to learn embeddings for the character segments, and construct word embedding by summing over the segment embeddings
- This resulted vector is competitive with GloVe in standard tasks, better in some languages

## Bias in Language

### Measure of Bias
- Bias: attitudes that affect our understanding, actions, and decisions in an unconscious manner
- Bias is generally measured using Implicit Association Tests (IATs):
    - Subjects asked to assign words to categories
    - Each category is consist of a word pair of target and attribute:
        - Example: [Flowers or Pleasant] and [Insects or Unpleasant] are two categories
        - Flowers is a target, and Pleasant is an attribute
        - Some word pairs are consistenent with stereotype, some are against stereotype
        - Example: [Flowers or Unpleasant] and [Insects or Pleasant]
    - Subjects need to assign new word (which is either a new attribute or a new target) to one category
    - Comparing reaction times across trials with different word pairs 
    - Subjects tend to be slower and more error-prone in assignments against stereotype
    - IAT score = difference in reaction time between stereotype-consistent and stereotype-inconsistent trials

### Caliskan, Bryson, and Narayanan (Science 2017)
- First identify a spectrum of known biases, as measured by the Implicit Association Test
- The known bias is represented through target-attribute sets
    - e.g. Set of Target Name [Flowers, Insects]
    - e.g. Set of Attribute Name [Pleasant, Unpleasant]
    - Flowers: {aster, clover, hyacinth, ...} 25 words
    - Insects: {ant, caterpillar, flea, ...} 25 words
    - Pleasant: {caress, freedom, health, ...} 25 words
    - Unpleasant: {abuse, crash, filth, ...} 25 words
- The paper measures bias as (e.g.):
    - [(average cosine similairty between Flowers Target to Pleasant Attributes) - (average cosine similairty between Flowers Target to Unpleasant Attributes)]
    - minus
    - [(average cosine similairty between Insects Target to Pleasant Attributes) - (average cosine similairty between Insects Target to Unpleasant Attributes)]
- The bias they found is similar to what IAT idetified (almost all IAT identified bias are also significant in this word-embedding based bias measure, same in direction and similar in size)

### Garg, Schiebinger, Jurafsky, and Zou (PNAS 2018)
- Use word embeddings to construct similar bias measure but between gender and occupations
- Found a strong correlation between empirical women occupation difference and the bias measure

## Linguistic

### Introduction to Linguistic Annotation
- The models we have seen so far have counted words and phrases, or embedded sequences based on local ordering of words
- Local ordering basically means proximity of one word to another
- But count or only word ordering can fails:
    - Consider this example: “The defendant was negligent” and “The defendant was not negligent”
    - In the first case, a simple proximity based model can say that negligent occurs near defendant, so that probably an attribute
    - In the second case, this model will fail: although negligent still occur near defendant, "no" reverse the attribute
    - Hence, we can see that models based on local order cannot identify attribute of targets correctly in all cases
- This motivates using information on sentence grammar: linguistic annotations

### Dependency Grammar
- Syntactic structure consists of words, linked by binary directed relations called dependencies
- Dependencies identify the grammatical relations between words
- Dependencies:
    - Directed arcs indicate dependencies: a one-way link from a “head” token to a “dependent” token
    - A word can be head multiple times, but dependent only one
- Dependency tree:
    - Example: "I have a dog"
    - have is head of I and dog (the dependency tree is 1 layer)
    - Example: "I have a white dog"
    - have is head of I and dog, dog is head of white (the dependency tree is 2 layer)
    - For complex sentence, the dependency tree will be deeper
    - Dependency tree are mostly determined by the ordering of POS tags
    - The “root” of a sentence is the main verb 
- Arc labels of dependency indicate functional relations:
    - Some examples:
    - nsubj: verb -> subject doing the verb
    - dobj: verb -> object targeted by the verb
    - amod: noun -> attribute of the noun

### Functional Relations
- Subjects:
    - nsubj: nominal subject
        - Non-clausal constituent in the subject position of an active verb
        - Example: Clinton defeated Dole
        - Clinton <-nsubj-  defeated
    - nsubjpass: passive nominal subject
        - Non-clausal constituent in the subject position of a passive verb
        - Example: Dole was defeated by Clinton
        - Dole <-nsubjpass-  defeated
- Objects:
    - dobj: direct object
        - Noun phrase, the (accusative) object of the verb
        - Example: She gave me a raise
        - gave -dobj-> raise
    - dative: dative or indirect object
        - Noun phrase, the (dative) object of the verb
        - Example: She gave me a raise
        - gave -dative-> me
    - pobj: object of a preposition
        - Noun phrase following a preposition
        - Example: I sat on the chair
        - on -pobj-> chair
- Adjectives:
    - acomp: adjectival complement
        - adjectival phrase which functions as object of verb
        - Example: Bill is honest
        - is -acomp-> honest
    - amod: adjectival modifier
        - modifies the meaning of the noun phrase
        - Example: Sam eats red meat
        - meat -amod-> red
    - appos: appositional modifier
        - is a noun phrase giving additional information of the preceding noun phrase
- Verbs:
    - aux: auxiliary
        - links between a verb and helping verb, including modals
        - Example: Reagan has died
        - aux(died -> has)
    - auxpass: passive auxiliary
        - links between a main verb and helping verb in passive constructions
        - Example: Laws have been broken
        - auxpass(broken -> been)
    - prt: phrasal verb particle
        - identifies a phrasal verb: links verb with its particle
        - Example: He pulls off his task
        - prt(pulls -> off)
- Etc:
    - neg: negation modifier
        - captures negation and the verb it modifies
        - Bill is not a scientist
        - neg(is -> not)
    - poss: possession modifier
        - holds between noun phrase and its possessive determiner, or genitive’s complement
        - Bill’s clothes
        - poss(clothes -> Bill)
        